# TDWI Lab 3 Part 3: PR Review Automations

In this lesson you will configure a **Cursor Automation** that reviews pull requests—and fixes **blocking errors**—after you mark a PR ready for review. You will then run a **Cloud Agent** to add a Streamlit **Revenue Explorer** and observe the full handoff: draft implementation PR → human review → ready → automation review (and optional fix draft PR).

**Lab design note:** You are learning the general **agent-as-reviewer** pattern and how **Automations** wire it on GitHub. Cursor also ships dedicated features for this—**Bugbot** and **Approval Agents**—and other AI tools have equivalents. In production, explore those built-ins first; we use a custom Automation here so the pattern is portable and not tied to one product feature.

## Learning Objectives

By the end of this mini-lesson you will be able to:
- Create a GitHub PR-triggered Cursor Automation with **review and fix** behavior
- Use the **Pull request opened** trigger (ready PRs only—not draft creation) to avoid automation loops
- Configure automation output: PR comments plus **draft** fix PRs for blocking errors only
- Run a Cloud Agent to add a Streamlit Revenue Explorer and walk through the full draft → ready → review (→ optional fix) flow
- Contrast a **custom Automation** with provider-native review (e.g. Cursor **Bugbot**, **Approval Agents**) and explain why the lab uses the general pattern

## Prerequisites

- Completed [README.md](README.md) setup (fork, clone, local `.venv`, test push)
- Completed [LAB3-Part-1-Cloud-Agent-Environment-Setup.ipynb](LAB3-Part-1-Cloud-Agent-Environment-Setup.ipynb) (env/secrets on your fork)
- Completed [LAB3-Part-2-Running-Cursor-Cloud-Agents.ipynb](LAB3-Part-2-Running-Cursor-Cloud-Agents.ipynb): pipeline fixes merged to `main`, tests green
- Cursor plan with **Automations** enabled; GitHub connected to **your fork**

**Important:** Configure the automation on the **same GitHub repo where you open PRs** (your fork). Each student sets up their own automation, unless the instructor demos on a shared fork.

## Step 1: Understand the workflow

Lab 3 uses a deliberate handoff between humans and agents:

1. **Cloud Agent** implements code and opens a **draft PR** (Step 4).
2. **You** pull the branch, run tests locally, and review the diff (Step 5).
3. **You** click **Ready for review** when you want automated review (Step 6).
4. **Cursor Automation** reviews the PR, posts comments, and—if it finds **blocking errors only**—may implement fixes and open a **new draft PR**.
5. **You** optionally review the fix PR and mark it **Ready for review** if you want another automated pass, or merge when satisfied.

Cursor defines two related triggers ([Automations docs](https://cursor.com/docs/cloud-agent/automations)):

| Trigger | When it fires |
|---------|----------------|
| **Draft opened** | A draft PR is created |
| **Pull request opened** | A non-draft PR is created **or** a draft is **marked ready for review** |

**For this lab, use Pull request opened only** (do **not** use **Draft opened**). Implementation and fix PRs stay **drafts** until you mark them ready—so the automation does not re-fire on its own fix PRs.

**Loop prevention (important):**

| Setup | Risk |
|-------|------|
| Trigger on **draft opened** + agent fixes issues | High—the automation re-fires on every new draft |
| Trigger on ready + fix **all** suggestions | High—there are always more suggestions to fix |
| **This lab:** ready only + fix **blocking errors** + open **draft** fix PR | Low—you control when the next pass runs |

Prompts are not deterministic; this is a tested best practice, not a guarantee.

## Step 2: Create the PR review automation

**Why build your own?** This step teaches the portable workflow—trigger, instructions, output—not the only way to get AI review on PRs. For day-to-day work on Cursor, **Bugbot** is an excellent default; **Approval Agents** are another productized option. After the lab, try those alongside or instead of a custom Automation. See [WORKFLOW_RECIPES.md](WORKFLOW_RECIPES.md) Recipe 5.

1. Open Cursor **Settings** → **Automations** (or the Automations section on [cursor.com](https://cursor.com)). See the [Automations documentation](https://cursor.com/docs/cloud-agent/automations) if the UI differs slightly.
2. Click **Create automation** (or equivalent).
3. **Trigger:** GitHub → **Pull request opened**.
4. **Repository:** Select **your fork** of this starter repo (e.g. `your-username/tdwi-agentic-sales-pipeline-starter`).
5. **Tools / output:** Enable **Comment on pull request** and any repo/PR tools the UI requires to open a **draft PR** when blocking fixes are needed.
6. Paste the following into the automation **instructions** field (general checklist—reuse on any PR in this repo):

```text
Review this pull request.

Focus on:
- Summary of what changed and whether the approach fits the existing codebase
- Correctness, edge cases, and error handling in the diff
- Whether tests were added or updated; note if the PR does not mention test results
- New or changed dependencies (e.g. requirements.txt): necessity and version pinning
- Scope: flag unrelated refactors or drive-by changes
- Security or data-handling concerns if relevant

Post a concise review as PR comments: summary, strengths, blocking errors, suggested improvements. Do not merge or approve. If you find blocking errors, implement the fixes and open a draft PR. The PR must be a draft.
```

7. Save the automation.

## Step 3: Save and verify the automation

- Confirm the automation appears in your Automations list and is enabled for your fork.
- Use the dashboard **run history** (if available) after Step 6 to confirm it executed.

**Note:** If you already marked a Part 2 PR as ready, toggling ready again may not re-fire the trigger. Part 3’s new PR (Step 4) is the intended test.

## Step 4: Cloud Agent — add Revenue Explorer

Your fork should already have the [`AGENTS.md`](AGENTS.md) you updated in Part 2 on GitHub. The Cloud Agent reads that file automatically—you do **not** need to repeat everything in this prompt.

**Already in `AGENTS.md` (confirm it is pushed to your fork):**
- Repo context and layout (`generate_sales_report.py`, tests, data paths)
- **Testing workflow** — run `python -m pytest test_sales_report.py`, iterate until green before pushing
- **Lab boundaries** — do not modify README or lab notebooks

This prompt states only **what to build** for Part 3. If you changed `AGENTS.md` locally since Part 2, commit and push before starting the agent (same as Part 2 Step 4).

Start a Cloud Agent on **your fork** (same Dockerfile-managed environment as Part 1). Go to [cursor.com/agents](https://cursor.com/agents) or the Agents window in Cursor, select your repository, and paste this prompt:

```text
Add a small Streamlit app called revenue_explorer.py at the repo root.

Product requirements:
- Sidebar: date range filter, multi-select product, optional customer_id filter.
- Main area: KPIs (total revenue, order count, average order value) for the filtered data.
- One chart: daily revenue trend for the filtered data.

Do not duplicate logic. Write clean, well-organized code.
Open a PR when done.
```

### Watch the agent and open the PR on GitHub

1. In the [Agents dashboard](https://cursor.com/agents) (or the Agents panel in Cursor), open your session and follow progress until the run **finishes**—edits, terminal output, pytest runs. This may take several minutes.
2. When the agent completes, it will usually open a **draft PR** on your fork. That is expected; you do not need a non-draft PR at this step.
3. On GitHub, open **your fork** → **Pull requests** → the agent's PR (often a `cursor/...` branch into `main`). Confirm it shows **Draft**.
4. Continue to **Step 5** for local review. You will mark the PR **Ready for review** in Step 6 to trigger your automation.

## Step 5: Human review (same pattern as Part 2)

1. From the draft PR you opened in Step 4, note the **branch name**.
2. Locally, fetch and check out the agent branch:

```bash
git fetch origin
git checkout <agent-branch-name>
```

3. With `.venv` activated, run tests:

```bash
python -m pytest test_sales_report.py
```

4. Optional: run the new app locally (install deps if the agent added `streamlit`):

```bash
pip install -r requirements.txt
streamlit run revenue_explorer.py
```

5. Read the code changes. Do **not** merge yet—you will trigger the automation in the next step.

## Step 6: Mark the PR ready for review

After your local review in Step 5, return to the **same PR** on GitHub.

1. Scroll to the **bottom** of the PR page and click **Ready for review** (shown on draft PRs only).
2. This fires **Pull request opened**—your review automation should start within a few minutes. You can also watch run status in the Cursor **Automations** run history (Step 3).

### When the automation finishes

Return to this PR on GitHub and review what the automation did:

1. Open the **Conversation** tab. Read the automation’s **review comments** (summary, strengths, blocking errors, suggestions).
2. Check whether it opened a **fix draft PR**. Your automation instructions tell it to do this only when it finds **blocking errors**—if the PR looked fine, you may see comments only and no new PR. That is normal.
3. If a fix draft PR exists, it is usually linked in the review comment or listed under **Pull requests** on your fork (new `cursor/...` or similar branch). It stays a **draft** and does **not** re-trigger your automation until you mark it ready.
4. **Optional: another review round.** After you review the fix locally (or skim the diff on GitHub), open the **fix PR**, scroll to the bottom, and click **Ready for review**. That triggers the same **review and fix** automation again—comments on the fix PR, and possibly another fix draft PR if it still finds blocking errors. You control when each round runs.

**Optional next steps:**

- Check out a fix branch locally, run tests, and review the diff before marking a fix PR ready.
- If **Bugbot** or **Approval Agents** are enabled, compare their output with your custom automation.
- Merge when you are satisfied (original PR, fix PR, or both—use your team’s workflow).

## Further reading: workflow framework

Parts 1–3 practiced pieces of a larger pattern: **probabilistic agents** for implementation, **deterministic scripts and CI** for gates, **fresh-context final review** (another agent—not the implementer) before humans merge. Review while building (tests, iteration) is expected; the framework adds a deliberate **final** critique—like asking a colleague with fresh eyes.

See [WORKFLOW_RECIPES.md](WORKFLOW_RECIPES.md) for the full framework, **[Harness vs team workflow](WORKFLOW_RECIPES.md#harness-vs-team-workflow)** (planning, tool loops, and what you still wire explicitly), the **Separate implementer from final reviewer** principle, why gates are **intentional but optional**, **provider-native review vs custom Automations**, and example recipes (check script → `AGENTS.md` → `/commit-code` → GitHub Actions → PR Automations → ticket-driven Cloud Agents).


## Debrief questions

1. Why use a **draft PR** for implementation agents and **ready for review** for the automation?
2. Why does this lab fix **blocking errors only** (not every suggestion)? What can go wrong if you trigger on **draft opened** and ask the automation to fix all suggestions?
3. What did your automation catch that you would have missed? What did it miss?
4. When would you use a **custom Automation** vs **Bugbot** vs **Approval Agents** vs both?
5. How could you add a **CI completed** trigger so review runs only after green checks?
6. Why does the lab teach custom Automations if vendor built-ins exist?